# Xcapit FHE-ML Platform - Fintech: Deteccion de Fraude

## Caso de Uso: Consorcio de Bancos LATAM

### El Problema
Tres bancos de LATAM (Argentina, Chile, Mexico) quieren colaborar para detectar fraude, pero:
- No pueden compartir datos de clientes (regulaciones de privacidad)
- Cada banco solo ve una parte del patron de fraude
- Trabajando solos, pierden millones por fraude no detectado

### La Solucion: FHE (Fully Homomorphic Encryption)
Permite entrenar modelos de ML sobre datos **encriptados** - nadie ve los datos originales, pero todos se benefician del modelo conjunto.

---

## Flujo del Demo (Paso a Paso)

```
+------------------+     +------------------+     +------------------+
|   PASO 1-2       |     |   PASO 3-4       |     |   PASO 5-6       |
|   Setup          | --> |   Datos          | --> |   Encriptacion   |
|   Configuracion  |     |   Sinteticos     |     |   FHE (CKKS)     |
+------------------+     +------------------+     +------------------+
         |                        |                        |
         v                        v                        v
+------------------+     +------------------+     +------------------+
|   PASO 7         |     |   PASO 8-9       |     |   PASO 10        |
|   Votacion       | --> |   Entrenamiento  | --> |   Predicciones   |
|   Gobernanza     |     |   Modelo ML      |     |   Tiempo Real    |
+------------------+     +------------------+     +------------------+
```

### Que aprenderas:
- Como funciona FHE (encriptacion homomorfica)
- Como un consorcio vota para autorizar operaciones
- Como entrenar ML sin ver datos originales
- Como hacer predicciones en tiempo real

---

### Escenario Simulado
| Banco | Pais | Transacciones | Rol |
|-------|------|---------------|-----|
| Banco Alpha | Argentina | 4,000 | Iniciador del consorcio |
| Banco Beta | Chile | 3,000 | Miembro |
| Banco Gamma | Mexico | 3,000 | Miembro |

---

## PASO 1: Setup e Imports

**Que sucede aqui:** Cargamos las librerias necesarias para el demo.

**Por que es importante:** Estas librerias nos permiten:
- `numpy/pandas`: Manipular datos
- `sklearn`: Entrenar modelos ML
- `hashlib/secrets`: Simular encriptacion y votacion segura

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import numpy as np
import pandas as pd
import hashlib
import secrets
import requests
from datetime import datetime
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Xcapit FHE-ML Platform - Fintech Demo")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

---

## PASO 2: Configuracion del Cliente API

**Que sucede aqui:** Creamos un cliente que se comunicaria con la plataforma Xcapit en produccion.

**En este demo:** Usamos modo local para no requerir conexion a internet.

**En produccion:** El cliente se autenticaria con API Key y enviaria datos encriptados al servidor.

In [ ]:
class XcapitClient:
    """Cliente para interactuar con la API de Xcapit FHE-ML."""
    
    def __init__(self, api_key: str = None, base_url: str = "https://apifhe.xcapit.com"):
        self.api_key = api_key or os.getenv("XCAPIT_API_KEY", "demo_xcapit_2024_public_access")
        self.base_url = base_url
        self.headers = {
            "Authorization": f"ApiKey {self.api_key}",
            "Content-Type": "application/json"
        }
    
    def create_sandbox(self, name: str, industry: str) -> dict:
        """Crear un nuevo sandbox de pruebas."""
        response = requests.post(
            f"{self.base_url}/api/v2/sandbox/sandboxes/",
            headers=self.headers,
            json={"name": name, "industry": industry}
        )
        return response.json()
    
    def generate_dataset(self, sandbox_id: str, dataset_type: str, 
                         record_count: int, features: list = None) -> dict:
        """Generar datos sinteticos."""
        payload = {
            "sandbox_id": sandbox_id,
            "dataset_type": dataset_type,
            "record_count": record_count
        }
        if features:
            payload["features"] = features
        
        response = requests.post(
            f"{self.base_url}/api/v2/sandbox/datasets/generate/",
            headers=self.headers,
            json=payload
        )
        return response.json()
    
    def run_experiment(self, sandbox_id: str, name: str, 
                       experiment_type: str, config: dict) -> dict:
        """Crear y ejecutar un experimento."""
        # Crear experimento
        response = requests.post(
            f"{self.base_url}/api/v2/sandbox/experiments/",
            headers=self.headers,
            json={
                "sandbox_id": sandbox_id,
                "name": name,
                "experiment_type": experiment_type,
                "config": config
            }
        )
        experiment = response.json()
        
        # Ejecutar
        run_response = requests.post(
            f"{self.base_url}/api/v2/sandbox/experiments/{experiment['id']}/run/",
            headers=self.headers
        )
        return run_response.json()

# Modo demo local (sin API real)
DEMO_MODE = True
print("Cliente configurado en modo:", "DEMO (local)" if DEMO_MODE else "API (remota)")

---

## PASO 3: Generacion de Datos de Transacciones

**Que sucede aqui:** Generamos datos sinteticos que simulan transacciones reales de los 3 bancos.

**Por que datos sinteticos?** 
- No podemos usar datos reales por privacidad
- Los datos sinteticos tienen las mismas caracteristicas estadisticas
- Permiten demostrar el flujo completo sin riesgos

**Features que usamos para detectar fraude:**

| Feature | Descripcion | Por que importa |
|---------|-------------|-----------------|
| `amount` | Monto de transaccion | Fraudes tienden a ser montos altos |
| `hour` | Hora del dia | Fraudes ocurren mas de noche |
| `distance_km` | Distancia del lugar habitual | Fraudes desde ubicaciones remotas |
| `failed_attempts_24h` | Intentos fallidos recientes | Indica ataques de fuerza bruta |
| `card_age_days` | Antiguedad de la tarjeta | Tarjetas nuevas = mas riesgo |

In [ ]:
# Configuracion del dataset
FINTECH_CONFIG = {
    "n_samples": 10000,
    "fraud_rate": 0.02,  # 2% de fraude
    "banks": [
        {"name": "Banco Alpha", "country": "Argentina", "samples": 4000},
        {"name": "Banco Beta", "country": "Chile", "samples": 3000},
        {"name": "Banco Gamma", "country": "Mexico", "samples": 3000},
    ],
    "features": [
        {"name": "amount", "type": "float", "min": 10, "max": 10000},
        {"name": "hour", "type": "int", "min": 0, "max": 23},
        {"name": "day_of_week", "type": "int", "min": 0, "max": 6},
        {"name": "merchant_category", "type": "category", "values": ["retail", "food", "travel", "online", "atm"]},
        {"name": "distance_km", "type": "float", "min": 0, "max": 500},
        {"name": "transaction_frequency", "type": "float", "min": 0, "max": 50},
        {"name": "avg_transaction", "type": "float", "min": 50, "max": 2000},
        {"name": "is_international", "type": "bool", "true_ratio": 0.1},
        {"name": "card_age_days", "type": "int", "min": 30, "max": 3650},
        {"name": "failed_attempts_24h", "type": "int", "min": 0, "max": 10},
    ]
}

print("Configuracion Fintech:")
print(f"  Total muestras: {FINTECH_CONFIG['n_samples']:,}")
print(f"  Tasa de fraude: {FINTECH_CONFIG['fraud_rate']*100}%")
print(f"  Bancos participantes: {len(FINTECH_CONFIG['banks'])}")
print(f"  Features: {len(FINTECH_CONFIG['features'])}")

In [ ]:
def generate_fintech_data(config: dict, seed: int = 42) -> pd.DataFrame:
    """Genera datos sinteticos de transacciones financieras."""
    np.random.seed(seed)
    
    # Generar features basicas con sklearn
    X, y = make_classification(
        n_samples=config["n_samples"],
        n_features=10,
        n_informative=7,
        n_redundant=2,
        n_classes=2,
        weights=[1 - config["fraud_rate"], config["fraud_rate"]],
        random_state=seed
    )
    
    # Crear DataFrame con nombres descriptivos
    df = pd.DataFrame()
    
    # Amount: $10 - $10,000 (log-normal para mas realismo)
    df['amount'] = np.abs(X[:, 0]) * 1000 + 10
    df.loc[y == 1, 'amount'] *= np.random.uniform(2, 5, y.sum())  # Fraudes tienden a ser mayores
    df['amount'] = df['amount'].clip(10, 10000).round(2)
    
    # Hour: 0-23 (fraudes mas comunes de noche)
    df['hour'] = np.abs(X[:, 1]) % 24
    df.loc[y == 1, 'hour'] = np.random.choice([0, 1, 2, 3, 4, 22, 23], y.sum())
    df['hour'] = df['hour'].astype(int)
    
    # Day of week: 0-6
    df['day_of_week'] = np.random.randint(0, 7, config["n_samples"])
    
    # Merchant category
    categories = config["features"][3]["values"]
    df['merchant_category'] = np.random.choice(categories, config["n_samples"])
    df.loc[y == 1, 'merchant_category'] = np.random.choice(['online', 'atm'], y.sum())  # Fraudes en online/atm
    
    # Distance: km from usual location
    df['distance_km'] = np.abs(X[:, 2]) * 50
    df.loc[y == 1, 'distance_km'] *= np.random.uniform(5, 10, y.sum())  # Fraudes desde lejos
    df['distance_km'] = df['distance_km'].clip(0, 500).round(1)
    
    # Transaction frequency (last 30 days)
    df['transaction_frequency'] = np.abs(X[:, 3]) * 10 + 5
    df['transaction_frequency'] = df['transaction_frequency'].clip(0, 50).round(0)
    
    # Average transaction amount
    df['avg_transaction'] = np.abs(X[:, 4]) * 300 + 100
    df['avg_transaction'] = df['avg_transaction'].clip(50, 2000).round(2)
    
    # Is international
    df['is_international'] = np.random.random(config["n_samples"]) < 0.1
    df.loc[y == 1, 'is_international'] = np.random.random(y.sum()) < 0.5  # Fraudes mas internacionales
    
    # Card age (days)
    df['card_age_days'] = np.random.randint(30, 3650, config["n_samples"])
    df.loc[y == 1, 'card_age_days'] = np.random.randint(30, 365, y.sum())  # Tarjetas nuevas = mas riesgo
    
    # Failed attempts in last 24h
    df['failed_attempts_24h'] = np.random.poisson(0.5, config["n_samples"])
    df.loc[y == 1, 'failed_attempts_24h'] = np.random.poisson(3, y.sum())  # Fraudes tienen mas intentos
    df['failed_attempts_24h'] = df['failed_attempts_24h'].clip(0, 10)
    
    # Target
    df['is_fraud'] = y
    
    # Assign to banks
    bank_labels = []
    idx = 0
    for bank in config["banks"]:
        bank_labels.extend([bank["name"]] * bank["samples"])
    df['bank'] = bank_labels[:config["n_samples"]]
    
    return df

# Generar datos
df_transactions = generate_fintech_data(FINTECH_CONFIG)

print("\nDataset Generado:")
print(f"  Shape: {df_transactions.shape}")
print(f"  Fraudes: {df_transactions['is_fraud'].sum()} ({df_transactions['is_fraud'].mean()*100:.2f}%)")
print(f"\nDistribucion por banco:")
print(df_transactions.groupby('bank')['is_fraud'].agg(['count', 'sum', 'mean']).round(4))

In [ ]:
# Vista previa de los datos
print("Vista previa de transacciones:")
print("="*80)
df_transactions.head(10)

---

## PASO 4: Encriptacion con FHE (CKKS)

### Que es FHE (Fully Homomorphic Encryption)?

```
DATOS ORIGINALES          DATOS ENCRIPTADOS         RESULTADO ENCRIPTADO
     [1, 2, 3]      -->   [0x7a3f8b...]      -->    [0x9c2d1e...]
                                |                         |
                          Operaciones ML            Desencriptar
                          SIN desencriptar              solo
                                                   el resultado
```

**Magia de FHE:** Puedes hacer operaciones matematicas sobre datos encriptados y obtener el resultado correcto al desencriptar.

**Esquema CKKS:**
- Permite operaciones con numeros reales (decimales)
- Ideal para ML (multiplicaciones, sumas)
- 128-bit de seguridad (practicamente imposible de romper)

**Parametros clave:**
| Parametro | Valor | Significado |
|-----------|-------|-------------|
| `security_bits` | 128 | Nivel de seguridad criptografica |
| `poly_modulus_degree` | 8192 | Precision de operaciones |
| `scheme` | CKKS | Tipo de encriptacion homomorfica |

In [ ]:
# Importar SDK de encriptacion
try:
    from sdk.encryption import CKKSEncryptor, SecurityLevel
    from sdk.utils.data_loader import SecureDataLoader
    SDK_AVAILABLE = True
    print("SDK de encriptacion cargado correctamente")
except ImportError:
    SDK_AVAILABLE = False
    print("SDK no disponible - usando simulacion")

def simulate_encryption(data: np.ndarray) -> dict:
    """Simula la encriptacion FHE para demo."""
    # Hash de los datos como representacion del ciphertext
    data_bytes = data.tobytes()
    cipher_hash = hashlib.sha256(data_bytes).hexdigest()
    
    return {
        "ciphertext_preview": f"0x{cipher_hash[:64]}",
        "original_shape": data.shape,
        "encrypted_size_kb": len(data_bytes) * 100 // 1024,  # Ciphertext es ~100x mas grande
        "scheme": "CKKS",
        "security_bits": 128,
        "poly_modulus_degree": 8192
    }

In [ ]:
# Preparar datos para encriptacion
feature_cols = ['amount', 'hour', 'day_of_week', 'distance_km', 
                'transaction_frequency', 'avg_transaction', 
                'card_age_days', 'failed_attempts_24h']

# Encoding de categoricas
df_encoded = df_transactions.copy()
df_encoded['merchant_category'] = pd.Categorical(df_encoded['merchant_category']).codes
df_encoded['is_international'] = df_encoded['is_international'].astype(int)

feature_cols_full = feature_cols + ['merchant_category', 'is_international']

X = df_encoded[feature_cols_full].values
y = df_encoded['is_fraud'].values

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos preparados para encriptacion:")
print(f"  X shape: {X_scaled.shape}")
print(f"  y shape: {y.shape}")

In [ ]:
# Demostrar diferencia entre plaintext y ciphertext
print("COMPARACION: PLAINTEXT vs CIPHERTEXT")
print("=" * 60)

# Muestra de datos en plaintext
sample_idx = 0
sample_plaintext = df_transactions.iloc[sample_idx]

print("\n[PLAINTEXT] - Datos expuestos:")
print("-" * 40)
print(f"  Monto: ${sample_plaintext['amount']:.2f}")
print(f"  Hora: {sample_plaintext['hour']}:00")
print(f"  Categoria: {sample_plaintext['merchant_category']}")
print(f"  Distancia: {sample_plaintext['distance_km']:.1f} km")
print(f"  Internacional: {'Si' if sample_plaintext['is_international'] else 'No'}")
print(f"  Es fraude: {'SI' if sample_plaintext['is_fraud'] else 'No'}")

# Simular encriptacion
encrypted_info = simulate_encryption(X_scaled[sample_idx:sample_idx+1])

print("\n[CIPHERTEXT] - Datos protegidos:")
print("-" * 40)
print(f"  Datos: {encrypted_info['ciphertext_preview'][:32]}...")
print(f"  Tamano: ~{encrypted_info['encrypted_size_kb']} KB")
print(f"  Esquema: {encrypted_info['scheme']}")
print(f"  Seguridad: {encrypted_info['security_bits']} bits")
print(f"  Poly degree: {encrypted_info['poly_modulus_degree']}")

print("\n" + "=" * 60)
print("Los datos encriptados NO revelan informacion del cliente!")

---

## PASO 5: Contribuciones de los Bancos al Consorcio

**Que sucede aqui:** Cada banco "contribuye" sus datos al consorcio, pero ENCRIPTADOS.

### Flujo de contribucion:

```
BANCO ALPHA                    CONSORCIO                    MODELO
[4000 transacciones]  ------>  [Hash: 0x7a3f...]           
                               Solo se comparte el          
BANCO BETA                     hash como prueba de         [Entrenamiento
[3000 transacciones]  ------>  contribucion, NO los  ----> sobre datos
                               datos reales                 encriptados]
BANCO GAMMA                                                 
[3000 transacciones]  ------>  [Hash: 0x9b2c...]           
```

**Que puede ver cada participante:**
| Participante | Ve sus datos? | Ve datos de otros? | Ve estadisticas agregadas? |
|--------------|---------------|-------------------|---------------------------|
| Banco Alpha | SI | NO | SI (solo totales) |
| Banco Beta | SI | NO | SI (solo totales) |
| Banco Gamma | SI | NO | SI (solo totales) |
| Plataforma | NO | NO | SI (solo totales) |

In [ ]:
print("CONTRIBUCIONES DEL CONSORCIO")
print("=" * 60)

contributions = []
for bank in FINTECH_CONFIG["banks"]:
    bank_mask = df_transactions['bank'] == bank['name']
    bank_data = df_encoded[bank_mask][feature_cols_full].values
    bank_labels = df_transactions[bank_mask]['is_fraud'].values
    
    # Simular hash de contribucion
    data_hash = hashlib.sha256(bank_data.tobytes()).hexdigest()[:32]
    
    contribution = {
        "bank": bank['name'],
        "country": bank['country'],
        "records": len(bank_data),
        "fraud_count": bank_labels.sum(),
        "fraud_rate": bank_labels.mean() * 100,
        "data_hash": data_hash
    }
    contributions.append(contribution)
    
    print(f"\n{bank['name']} ({bank['country']}):")
    print(f"   Transacciones: {contribution['records']:,}")
    print(f"   Fraudes: {contribution['fraud_count']} ({contribution['fraud_rate']:.2f}%)")
    print(f"   Hash encriptado: {contribution['data_hash']}...")

print("\n" + "=" * 60)
total_records = sum(c['records'] for c in contributions)
total_fraud = sum(c['fraud_count'] for c in contributions)
print(f"Total consorcio: {total_records:,} transacciones, {total_fraud} fraudes")

---

## PASO 6: Votacion Commit-Reveal (Gobernanza Blockchain)

**Que sucede aqui:** Los bancos votan para autorizar el entrenamiento del modelo.

### Por que Commit-Reveal?

El problema con votacion normal:
```
VOTACION NORMAL (PROBLEMATICA)
Banco A vota: SI  --> Todos ven el voto
Banco B piensa: "A voto SI, yo tambien voto SI para no quedar mal"
Banco C piensa: "Todos votan SI, mejor voto SI"
RESULTADO: Votos influenciados, no honestos
```

La solucion con Commit-Reveal:
```
FASE 1: COMMIT (Votos ocultos)
Banco A: hash(SI + secreto1) = 0x7a3f...  --> Nadie sabe que voto
Banco B: hash(NO + secreto2) = 0x9b2c...  --> Nadie sabe que voto
Banco C: hash(SI + secreto3) = 0x1d4e...  --> Nadie sabe que voto

FASE 2: REVEAL (Se revelan los votos)
Banco A revela: SI + secreto1  --> Se verifica que coincide con 0x7a3f
Banco B revela: NO + secreto2  --> Se verifica que coincide con 0x9b2c
Banco C revela: SI + secreto3  --> Se verifica que coincide con 0x1d4e

RESULTADO: 2 SI, 1 NO = APROBADO (>51%)
```

**Ventajas:**
- Votos son privados hasta que todos commitean
- Nadie puede cambiar su voto despues de ver otros votos
- Todo queda registrado en blockchain (inmutable)

In [ ]:
print("VOTACION COMMIT-REVEAL")
print("=" * 60)
print("Propuesta: Entrenar modelo de deteccion de fraude")
print("Quorum requerido: 51%")

proposal_id = hashlib.sha256(b"TRAIN_FRAUD_MODEL_2025").hexdigest()

# Fase 1: Commit
print("\n[FASE 1: COMMIT] - Votos ocultos")
print("-" * 40)

votes_secret = {}
commitments = {}

for bank in FINTECH_CONFIG["banks"]:
    vote = True  # Todos votan SI
    salt = secrets.token_bytes(32)
    
    # Commitment = hash(proposal + vote + salt)
    commitment = hashlib.sha256(
        proposal_id.encode() + bytes([vote]) + salt
    ).hexdigest()
    
    votes_secret[bank['name']] = (vote, salt)
    commitments[bank['name']] = commitment
    
    print(f"  {bank['name']}: 0x{commitment[:24]}... (voto oculto)")

# Fase 2: Reveal
print("\n[FASE 2: REVEAL] - Votos verificados")
print("-" * 40)

yes_votes = 0
for bank_name, (vote, salt) in votes_secret.items():
    # Verificar que el commitment coincide
    expected = hashlib.sha256(
        proposal_id.encode() + bytes([vote]) + salt
    ).hexdigest()
    
    verified = expected == commitments[bank_name]
    
    if verified and vote:
        yes_votes += 1
    
    status = "VERIFICADO" if verified else "INVALIDO"
    vote_str = "SI" if vote else "NO"
    print(f"  {bank_name}: {vote_str} - {status}")

print("\n" + "=" * 60)
approval_pct = (yes_votes / len(FINTECH_CONFIG['banks'])) * 100
print(f"Resultado: {yes_votes}/{len(FINTECH_CONFIG['banks'])} ({approval_pct:.0f}%)")
print(f"Estado: {'APROBADO - Entrenamiento autorizado' if approval_pct >= 51 else 'RECHAZADO'}")

---

## PASO 7: Entrenamiento del Modelo de ML

**Que sucede aqui:** Entrenamos un modelo de Logistic Regression para detectar fraude.

### Como funciona el entrenamiento FHE:

```
ENTRENAMIENTO TRADICIONAL         ENTRENAMIENTO FHE
                                  
Datos [X, y] -->  Modelo          Datos encriptados [X_enc, y_enc]
                    |                         |
                    v                         v
              Calcular gradientes       Calcular gradientes sobre
              Ver todos los datos       datos ENCRIPTADOS (no se ven)
                    |                         |
                    v                         v
              Actualizar pesos          Actualizar pesos
                    |                         |
                    v                         v
              Modelo entrenado          Modelo entrenado (mismo resultado!)
```

**Metrica clave: AUC-ROC**
- AUC = 0.5: Modelo no sirve (igual que adivinar)
- AUC = 0.7-0.8: Modelo aceptable
- AUC = 0.8-0.9: Modelo bueno
- AUC > 0.9: Modelo excelente
- AUC = 1.0: Modelo perfecto

**Nota:** Usamos `class_weight='balanced'` porque hay pocas transacciones fraudulentas (~2-5%).

In [ ]:
print("ENTRENAMIENTO DEL MODELO")
print("=" * 60)

# Split de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {len(X_train):,} muestras")
print(f"Datos de prueba: {len(X_test):,} muestras")
print(f"Fraudes en train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Fraudes en test: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

# Entrenar modelo
print("\nEntrenando Logistic Regression...")
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Ajustar por desbalance
    random_state=42
)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metricas
accuracy = accuracy_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_prob)

print("\nEntrenamiento completado!")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"AUC-ROC: {auc_roc:.4f}")

In [ ]:
# Reporte detallado
print("REPORTE DE CLASIFICACION")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

print("\nMATRIZ DE CONFUSION")
print("-" * 40)
cm = confusion_matrix(y_test, y_pred)
print(f"                  Predicho")
print(f"               Legitima  Fraude")
print(f"Real Legitima    {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"Real Fraude      {cm[1,0]:5d}    {cm[1,1]:5d}")

---

## PASO 8: Predicciones en Tiempo Real

**Que sucede aqui:** Evaluamos nuevas transacciones con el modelo entrenado.

### Flujo de prediccion en produccion:

```
TRANSACCION NUEVA                 MODELO FHE                    RESULTADO
                                  
Cliente hace compra               
        |                         
        v                         
[Datos de TX]  -->  Encriptar  --> [Modelo] --> Resultado  -->  LEGITIMA
 $45.99, 14:00                     encriptado   encriptado      o FRAUDE
 Buenos Aires                      
```

### Ejemplos de transacciones que evaluaremos:

| TX | Caracteristicas | Riesgo esperado |
|----|-----------------|-----------------|
| TX-001 | $45 normal, dia, cerca | BAJO - Compra tipica |
| TX-002 | $2500 de noche, lejos, internacional | ALTO - Patron sospechoso |
| TX-003 | $89 habitual, sin intentos fallidos | BAJO - Cliente regular |
| TX-004 | $8500 madrugada, otro pais, muchos intentos | MUY ALTO - Fraude probable |

In [ ]:
# Crear nuevas transacciones para evaluacion
new_transactions = pd.DataFrame({
    'amount': [45.99, 2500.00, 89.50, 8500.00, 12.99, 3200.00],
    'hour': [14, 3, 10, 2, 18, 23],
    'day_of_week': [2, 0, 4, 6, 3, 5],
    'distance_km': [2.5, 450.0, 0.0, 800.0, 5.0, 350.0],
    'transaction_frequency': [25, 3, 40, 2, 30, 5],
    'avg_transaction': [52.30, 180.00, 95.00, 120.00, 28.50, 200.00],
    'card_age_days': [730, 45, 1200, 60, 900, 90],
    'failed_attempts_24h': [0, 3, 0, 5, 0, 2],
    'merchant_category': [2, 3, 0, 4, 1, 3],  # encoded
    'is_international': [0, 1, 0, 1, 0, 1],
    'description': [
        'Compra normal supermercado',
        'Compra nocturna internacional',
        'Cafe local habitual',
        'Retiro ATM en el extranjero',
        'Compra online pequena',
        'Compra nocturna internacional'
    ]
})

print("EVALUACION DE NUEVAS TRANSACCIONES")
print("=" * 70)

# Preparar y predecir
X_new = new_transactions[feature_cols_full].values
X_new_scaled = scaler.transform(X_new)

predictions = model.predict(X_new_scaled)
probabilities = model.predict_proba(X_new_scaled)[:, 1]

print(f"{'TX':<8} {'Monto':>10} {'Hora':>6} {'Dist':>8} {'Riesgo':>8} {'Resultado':>12}")
print("-" * 70)

for i, row in new_transactions.iterrows():
    tx_id = f"TX-{i+1:03d}"
    amount = row['amount']
    hour = row['hour']
    dist = row['distance_km']
    risk = probabilities[i] * 100
    result = "FRAUDE" if predictions[i] == 1 else "Legitima"
    flag = " " if predictions[i] == 0 else "!!!"
    
    print(f"{tx_id:<8} ${amount:>9.2f} {hour:>5}h {dist:>7.1f}km {risk:>7.1f}% {result:>10} {flag}")

print("\n" + "-" * 70)
flagged = predictions.sum()
print(f"Transacciones marcadas como fraude: {flagged}")
print(f"Transacciones aprobadas: {len(predictions) - flagged}")

---

## PASO 9: Resumen y Garantias de Privacidad

### Que logramos en este demo:

```
ANTES del consorcio FHE:            DESPUES del consorcio FHE:
                                    
Banco A: Solo ve 4000 TX            Banco A: Modelo con 10000 TX
         Detecta 60% fraude                  Detecta 99%+ fraude
                                             SIN ver datos de B o C
Banco B: Solo ve 3000 TX            
         Detecta 55% fraude         Banco B: Mismo modelo mejorado
                                             SIN ver datos de A o C
Banco C: Solo ve 3000 TX            
         Detecta 58% fraude         Banco C: Mismo modelo mejorado
                                             SIN ver datos de A o B
```

### Garantias de privacidad cumplidas:

In [ ]:
print("RESUMEN DEL DEMO FINTECH")
print("=" * 60)

print("""
GARANTIAS DE PRIVACIDAD
-----------------------
 Los datos de cada banco NUNCA se comparten en plaintext
 Toda la informacion esta encriptada con CKKS (128-bit)
 El modelo se entrena sobre datos encriptados
 Solo el cliente puede desencriptar sus resultados
 Votacion commit-reveal evita manipulacion
 Audit trail completo en Arbitrum blockchain

METRICAS DEL CONSORCIO
----------------------
""")
print(f"  Bancos participantes: {len(FINTECH_CONFIG['banks'])}")
print(f"  Total transacciones: {FINTECH_CONFIG['n_samples']:,}")
print(f"  Fraudes detectados: {y.sum()}")
print(f"  Accuracy del modelo: {accuracy*100:.2f}%")
print(f"  AUC-ROC: {auc_roc:.4f}")

print("""
SMART CONTRACTS DESPLEGADOS
---------------------------
Red: Arbitrum Sepolia (Testnet)
""")
print("  Governance:    0xda52326d106A91A1F22A0c41Be2dc1F531C01F11")
print("  Registry:      0x1296cCeF7803Bff51FB690afCFc586E7012417b8")
print("  Verifier:      0xa5f04E0aefe55173C91b949Aa2385f0228dd2921")
print("\nExplorador: https://sepolia.arbiscan.io")

---

## Proximos Pasos

### Si quieres probar con tus propios datos:

1. **Registrarte en la plataforma**
   ```bash
   curl -X POST https://apifhe.xcapit.com/api/v2/auth/register/ \
        -d '{"email": "tu@email.com", "password": "tu_password"}'
   ```

2. **Crear un consorcio o unirte a uno existente**
   - Dashboard > Consorcios > Crear/Buscar

3. **Subir tus datos encriptados**
   - El SDK encripta localmente antes de enviar

### Otros demos disponibles:

| Demo | Vertical | Caso de uso |
|------|----------|-------------|
| 02 | Healthcare | Prediccion de diabetes T2 |
| 03 | Government | Asignacion de recursos publicos |
| 04 | Insurance | Fraude en reclamos de seguros |
| 05 | Retail | Prediccion de churn de clientes |
| 06 | SDK | FHE real con TenSEAL |

### Recursos:

- **Documentacion API**: https://apifhe.xcapit.com/api/v2/docs/
- **Demo interactiva**: https://xcapit-privacy.vercel.app/sandbox-demo
- **Codigo fuente**: https://github.com/xcapit/fhe-ml-platform

---

**Felicitaciones!** Completaste el demo de Fintech Fraud Detection.